# Filter Designer

**Change the droop-correction filter without rebuilding anything.**

Move two sliders, look at the plot, press a button. The new filter is in the chip about a
second later. No Vivado, no bitstream, no reboot.

---

## Why this is possible

The filter is defined by 309 numbers held in a small memory inside the chip. Those numbers
were once fixed when the design was built — changing them meant a 25-minute rebuild on a PC
and reloading the board.

They are now **writable from software**, so this notebook can replace them directly.

## What you can and cannot change here

| | |
|---|---|
| **Passband** — how much of the window is corrected | ✅ here, instantly |
| **Stopband** — where the filter cuts off | ✅ here, instantly |
| **The window function** — trades band edge sharpness against stopband depth | ✅ here, instantly |
| **The number of taps (309)** | ❌ needs a rebuild — it is built into the wiring, not the numbers |

309 taps with adjustable passband and stopband covers nearly everything you would want. The
tap count only matters if you need the correction to work at decimation below 256, which needs
a shorter filter — see `README.md`.

## Making it permanent

Anything loaded here is lost when the board is power-cycled, because the chip reverts to the
coefficients built into the bitstream. The last section writes your design back to the source
files so the next build starts from it.

---

### Simulation or the real trap?

**Neither — this is a design tool, not a measurement.** It works out correction-filter
coefficients and loads them into the chip. Nothing here touches a coil, a signal or the
trap, so it is unchanged between emulation and real work.

The filter it designs *does* matter for real data: it flattens the averager's droop across
the band you measure in. If you move to a different frequency or averaging setting, come
back here and check the passband still covers what you use.


## Setup — run this first

In [ ]:
import os, mmap, struct, time
import numpy as np
import matplotlib.pyplot as plt

FS_CLK   = 125e6
ACQ_BASE = 0x40600000

R_CTRL, R_DEC, R_STATUS2 = 0x00, 0x08, 0x38
R_COEF_ADDR, R_COEF_DATA = 0x40, 0x44

# These MUST match the hardware. They are set when the bitstream is built and
# cannot be changed from here.
NTAPS, NHALF, COEF_FRAC = 309, 155, 17
NSTAGE = 4                       # decimator stages, fixed in the hardware

fd = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
mm = mmap.mmap(fd, 0x1000, offset=ACQ_BASE)

def wr(off, val): mm[off:off+4] = struct.pack('<I', val & 0xFFFFFFFF)
def rd(off):      return struct.unpack('<I', mm[off:off+4])[0]

print('connected to the acquisition system at 0x%08X' % ACQ_BASE)

# Does this bitstream support run-time coefficients? Older ones do not.
wr(R_COEF_ADDR, 154)
probe = rd(R_COEF_ADDR)
if probe != 154:
    print('')
    print('*** This bitstream does not support changing coefficients at run time.')
    print('    Rebuild with the current design and reload the board.')
else:
    print('run-time coefficient loading is supported by this bitstream')

---
## 1. What is in the chip right now?

Reads all 155 stored coefficients back out and works out what filter they make.

**This is worth doing before you change anything**, for two reasons. It tells you what you are
starting from, and it is the only way to know what the chip actually contains — as opposed to
what it was supposed to contain. Those are different things.

In [ ]:
def s32(v):
    """Turn a register value into a signed number.

    THE HARDWARE ALREADY SIGN-EXTENDS TO 32 BITS before handing the value back:

        sys_rdata <= {{14{coef_rdata[17]}}, coef_rdata};

    So a stored -1 arrives as 0xFFFFFFFF, not as 0x3FFFF. Undoing an 18-bit
    sign here instead of a 32-bit one turns every negative coefficient into a
    large positive number - which is exactly what happened the first time this
    notebook was run: 41 of 155 values "failed", and 41 is precisely how many
    negative coefficients the filter has.
    """
    return v - (1 << 32) if v & 0x80000000 else v


def read_coeffs():
    """Read the unique half from the chip and mirror it into the full filter."""
    half = []
    for k in range(NHALF):
        wr(R_COEF_ADDR, k)
        half.append(s32(rd(R_COEF_DATA)))
    half = np.array(half, dtype=np.int64)
    # taps 155..308 mirror 153..0
    return np.concatenate([half, half[:-1][::-1]])


def curves(h):
    """Work out what a set of coefficients does. Returns arrays for plotting."""
    u = np.linspace(0, 0.35, 400)
    k = np.arange(len(h)) - len(h) // 2
    fir = np.abs(np.exp(-2j * np.pi * np.outer(u, k)) @ h) / float(1 << COEF_FRAC)

    s = np.ones_like(u); nz = u != 0
    s[nz] = np.sin(np.pi * u[nz]) / (np.pi * u[nz])
    droop = np.abs(s) ** NSTAGE               # what the decimator does
    both  = droop * fir                       # what you actually get
    return u, droop, fir, both


def describe(h, label):
    """Print what a set of coefficients does. Returns nothing on purpose.

    Returning the arrays would make Jupyter echo several hundred numbers after
    every call, burying the three lines that matter. Use curves() if you want
    the data.
    """
    u, droop, fir, both = curves(h)
    dc = h.sum() / float(1 << COEF_FRAC)
    band = u <= 0.20
    flat = 20 * np.log10(both[band].max() / max(both[band].min(), 1e-12))

    print('%s' % label)
    print('   gain at the centre of the window : %.6f   (should be 1.000000)' % dc)
    print('   flatness across the window       : %.3f dB   (smaller is better)' % flat)
    print('   biggest boost applied            : %.2f dB' % (20*np.log10(fir[band].max())))


current = read_coeffs()
print('read %d coefficients from the chip' % NHALF)
print('')
describe(current, 'CURRENTLY IN THE CHIP:')
print('')
print('If the gain at the centre is not close to 1.000, the chip does not contain')
print('the filter it was supposed to. Load a fresh one below and the problem goes')
print('away - and you will know that was the cause.')

---
## 2. Design a new one

Two numbers control the shape. Both are given as a **fraction of the output sample rate** —
using fractions rather than hertz means the same design works at every decimation setting.

| Setting | What it does | Sensible range |
|---|---|---|
| **Passband** | How much of the window gets corrected. Bigger = more usable width | 0.10 – 0.25 |
| **Stopband** | Where the filter cuts off completely. Must be above the passband | passband + 0.05 upward |

**The trade-off:** a wider passband gives you more usable measurement window, but the filter
has to boost harder at the edge, and boosting amplifies noise along with signal. A stopband
close to the passband gives a sharper edge but more ripple.

Start with the defaults. They correct out to 20% of the window and flatten it to 0.04 dB.

In [ ]:
# ---------------------------------------------------------------------------
#  CHANGE THESE TWO NUMBERS, then run the cell.
# ---------------------------------------------------------------------------
PASSBAND = 0.20      # correct out to this fraction of the window
STOPBAND = 0.28      # fully cut off beyond here (must be > PASSBAND)
# ---------------------------------------------------------------------------

WINDOW = 'blackman'  # 'blackman' (deep stopband) or 'hamming' (sharper edge)


def design(passband, stopband, window='blackman', ntaps=NTAPS, nstage=NSTAGE):
    """Work out the 309 numbers that undo the decimator's droop.

    The decimator sags towards the edges of the window by a known shape. We ask
    for exactly the opposite shape across the passband, taper smoothly to zero
    across the transition, then turn that into a set of coefficients.
    """
    ngrid = 8192
    u = np.linspace(0.0, 0.5, ngrid // 2 + 1)

    s = np.ones_like(u); nz = u != 0
    s[nz] = np.sin(np.pi * u[nz]) / (np.pi * u[nz])
    droop = np.abs(s) ** nstage

    D = np.zeros_like(u)
    inband = u <= passband
    D[inband] = 1.0 / droop[inband]                 # the inverse of the sag

    # A smooth edge, not a cliff: a cliff would need an infinitely long filter.
    tr = (u > passband) & (u < stopband)
    x = (u[tr] - passband) / (stopband - passband)
    D[tr] = D[inband][-1] * 0.5 * (1 + np.cos(np.pi * x))

    h = np.fft.irfft(D, n=ngrid)
    h = np.roll(h, ngrid // 2)
    c, half = ngrid // 2, ntaps // 2
    h = h[c - half: c + half + 1]
    h = h * (np.blackman(ntaps) if window == 'blackman' else np.hamming(ntaps))
    h = h / h.sum()                                 # gain of 1 at the centre

    q = np.round(h * (1 << COEF_FRAC)).astype(np.int64)
    q[ntaps // 2] += (1 << COEF_FRAC) - q.sum()     # make the gain exactly 1

    lim = (1 << 17) - 1
    if np.abs(q).max() > lim:
        raise ValueError('a coefficient is too big for the hardware (%d > %d). '
                         'Try a narrower passband.' % (np.abs(q).max(), lim))
    return q


assert STOPBAND > PASSBAND, 'STOPBAND must be larger than PASSBAND'
new = design(PASSBAND, STOPBAND, WINDOW)

print('NEW DESIGN   passband %.2f   stopband %.2f   window %s' % (PASSBAND, STOPBAND, WINDOW))
print('')
describe(new, 'What it would do:')
u, droop, fir, both = curves(new)

In [ ]:
# --- see it -----------------------------------------------------------------
_, _, _, both_cur = curves(current)      # what is in the chip at the moment

fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))

ax[0].plot(u, 20*np.log10(droop), label='the decimator alone (the sag)')
ax[0].plot(u, 20*np.log10(np.maximum(fir, 1e-12)), label='the new filter')
ax[0].plot(u, 20*np.log10(np.maximum(both, 1e-12)), lw=2.5, label='the two together')
ax[0].axvspan(0, PASSBAND, color='0.9', zorder=0)
ax[0].set_xlabel('position in the window (fraction of the output rate)')
ax[0].set_ylabel('dB'); ax[0].set_ylim(-40, 10)
ax[0].set_title('New design.\nShaded = the part being corrected')
ax[0].legend(fontsize=8); ax[0].grid(True, alpha=0.4)

ax[1].plot(u, 20*np.log10(np.maximum(both_cur, 1e-12)), label='what is in the chip now')
ax[1].plot(u, 20*np.log10(np.maximum(both, 1e-12)), lw=2, label='the new design')
ax[1].axvspan(0, PASSBAND, color='0.9', zorder=0)
ax[1].axhline(0, color='0.6', lw=0.8)
ax[1].set_xlabel('position in the window (fraction of the output rate)')
ax[1].set_ylabel('dB'); ax[1].set_ylim(-3, 3)
ax[1].set_title('Before and after.\nFlat along zero is the goal')
ax[1].legend(fontsize=8); ax[1].grid(True, alpha=0.4)

plt.tight_layout(); plt.show()

---
## 3. Load it into the chip

Takes about a second.

**The order matters.** A single pass of the filter takes 155 clock ticks and reads the
coefficient memory throughout. Changing the numbers midway mixes two different filters and
produces a burst of nonsense. So the chain is switched off first, the coefficients written,
and then it is switched back on.

The cell restores whatever settings were active before, and verifies every coefficient by
reading it back.

In [ ]:
def load_coeffs(h):
    """Write a filter into the chip and check every value arrived."""
    saved_ctrl = rd(R_CTRL)

    wr(R_CTRL, 0)                                    # 1. stop the chain
    time.sleep(0.01)

    half = h[:NHALF]
    for k, v in enumerate(half):                     # 2. write the coefficients
        wr(R_COEF_ADDR, int(k))
        wr(R_COEF_DATA, int(v) & 0x3FFFF)            # 18 bits

    bad = []                                         # 3. check them
    for k, v in enumerate(half):
        wr(R_COEF_ADDR, int(k))
        got = s32(rd(R_COEF_DATA))               # 32-bit sign - see s32()
        if got != int(v):
            bad.append((k, int(v), got))

    wr(R_CTRL, saved_ctrl | 2)                       # 4. reset and restore
    wr(R_CTRL, saved_ctrl)
    return bad


t0 = time.time()
bad = load_coeffs(new)
print('wrote and verified %d coefficients in %.2f s' % (NHALF, time.time() - t0))
print('')

if not bad:
    print('   All %d values read back exactly as written.' % NHALF)
    print('   The new filter is live. Run Test Stage 5 to measure it.')
else:
    print('   %d values did NOT read back correctly:' % len(bad))
    for k, want, got in bad[:5]:
        print('      coefficient %3d : wrote %+7d, read %+7d' % (k, want, got))
    print('   The values in the chip do not match what was sent. Check the')
    print('   bitstream supports run-time coefficients (see the setup cell).')

print('')
describe(read_coeffs(), 'NOW IN THE CHIP:')

---
## 4. Put it back the way it was

If a design turns out worse, this restores the one the bitstream was built with.

In [ ]:
try:
    original = design(0.20, 0.28, 'blackman')        # the built-in design
    bad = load_coeffs(original)
    print('restored the original filter.' if not bad else 'restore FAILED - see above')
    print('')
    describe(read_coeffs(), 'NOW IN THE CHIP:')
except Exception as e:
    print('could not restore:', e)

---
## 5. Make it permanent

Anything loaded above is lost when the board is power-cycled — the chip goes back to the
coefficients that were built into the bitstream.

To make a design permanent, put your two numbers into the generator on the PC and rebuild:

1. Open `acquisition_custom/gen_fir_coeffs.py`
2. Set `PASSBAND` and `STOPBAND` near the top to the values you settled on here
3. Run `python3 gen_fir_coeffs.py`
4. In Vivado: `source C:/rp/prj/v0.94/Test_Project/build_project.tcl`
5. Load the new bitstream — see `BITSTREAM_WORKFLOW.pdf`

The cell below prints exactly what to paste, so you do not have to remember the numbers.

In [ ]:
print('Put these two lines into acquisition_custom/gen_fir_coeffs.py:')
print('')
print('    PASSBAND = %.2f' % PASSBAND)
print('    STOPBAND = %.2f' % STOPBAND)
print('')
print('then run it, rebuild, and reload the board.')
print('')
print('Until you do, this design lives only in the chip\'s memory and will be')
print('lost at the next power cycle.')

---
## What to remember

- **Passband and stopband** can be changed from here, instantly.
- **The number of taps** cannot — it is built into the wiring. Rebuild for that.
- **Nothing here survives a power cycle** until you put the numbers into the generator and
  rebuild.
- **Always look at section 1 first.** If the gain at the centre is not 1.000, the chip is not
  holding the filter it should be, and that alone explains a lot of odd results.